# Basement, garage, and a code that only looks numeric

On every training row the basement identity holds:

$$
\mathrm{BsmtFinSF1} + \mathrm{BsmtFinSF2} + \mathrm{BsmtUnfSF} = \mathrm{TotalBsmtSF}.
$$

Thirty-seven houses have no basement. Eighty-one have no garage, and those are exactly the rows with `GarageCars` equal to 0. The mean price by the number of garage cars is not a straight climb:

| Cars | Houses | Mean price |
|---:|---:|---:|
| 0 | 81 | about 103317 |
| 1 | 369 | about 128117 |
| 2 | 824 | about 183852 |
| 3 | 181 | about 309636 |
| 4 | 5 | about 192656 |

Four cars is five houses, and their mean sits below the three-car mean. A line that treats `GarageCars` as a pure quantity keeps climbing at 4. The table does not.

`MSSubClass` is the sharper version of the same mistake. The mean price of code 20 is about 185225, on 536 houses. Code 30, the next code up, averages about 95830 on 69 houses. Code 60 averages about 239949 on 299 houses. The codes are labels for a kind of dwelling and a period. Their numeric order is not the order of the price.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# Basement identity, garage-car means, and three subclass means.
from collections import defaultdict
basement_ok = 0
for row in train:
    parts = [
        int(row["BsmtFinSF1"]),
        int(row["BsmtFinSF2"]),
        int(row["BsmtUnfSF"]),
    ]
    if sum(parts) == int(row["TotalBsmtSF"]):
        basement_ok += 1
print("basement", basement_ok)
assert basement_ok == 1460
assert sum(row["BsmtQual"] == "NA" for row in train) == 37
cars = defaultdict(list)
for row in train:
    cars[int(row["GarageCars"])].append(int(row["SalePrice"]))
expected_cars = {0: 81, 1: 369, 2: 824, 3: 181, 4: 5}
for count, houses in expected_cars.items():
    print(count, len(cars[count]), sum(cars[count]) / len(cars[count]))
    assert len(cars[count]) == houses
assert sum(cars[4]) / 5 < sum(cars[3]) / 181
subclass = defaultdict(list)
for row in train:
    subclass[int(row["MSSubClass"])].append(int(row["SalePrice"]))
print("20", len(subclass[20]), sum(subclass[20]) / len(subclass[20]))
print("30", len(subclass[30]), sum(subclass[30]) / len(subclass[30]))
print("60", len(subclass[60]), sum(subclass[60]) / len(subclass[60]))
assert (len(subclass[20]), len(subclass[30]), len(subclass[60])) == (536, 69, 299)
assert sum(subclass[30]) / 69 < sum(subclass[20]) / 536 < sum(subclass[60]) / 299


basement 1460
0 81 103317.28395061729
1 369 128116.68834688347
2 824 183851.66383495147
3 181 309636.1215469613
4 5 192655.8
20 536 185224.81156716417
30 69 95829.72463768115
60 299 239948.5016722408


## Pitfall

Sorting `MSSubClass` and drawing a line through it reads "30" as more house than "20." Code 30 is the older one-story type, and its mean price is about half the mean of code 20.

## Summary

The basement columns add up on all 1460 rows. No garage is 81 houses. Four garage cars is a group of 5, with a lower mean than three cars. Subclass codes 20, 30, and 60 are labels, and 30 is the cheap one.
